# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lilitkarapetyanofficial/flyrank-ml-internship/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

**Lane: Refresh / Content Opportunity Scoring** (one of the four predefined lanes in
`docs/ml-intern-dataset-and-lane-guide.md`).

**Research question:** Which existing pages should a reviewer look at first for refresh,
expansion, or protection, given limited review capacity?

I'm picking this lane because the underlying decision is a ranking problem, not a single
yes/no prediction — a reviewer wants an ordered queue, not a verdict on one page at a time.
It's also the lane the starter repo is fully instrumented for: a transparent rule baseline and
a trained-model comparison already exist end to end in `outputs/model_report.md`, so I can
evaluate a real ranking against a real, honest baseline from week one instead of building both
from nothing. Compared with open-ended signal analysis (Lane 1) or archetype clustering
(Lane 3), this lane produces a concrete, decision-oriented artifact — a ranked list with reason
codes — that a reviewer with a fixed weekly capacity could act on immediately.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing simport pandas as pd

import pandas as pd
from pathlib import Path

def find_data_csv(filename="content_refresh_anonymized.csv"):
    """Locate the starter CSV regardless of the notebook's current working
    directory -- avoids the 'No such file or directory: data/raw/...' error
    that shows up if Colab's cwd isn't the repo root."""
    here = Path.cwd()
    for base in [here, *here.parents]:
        candidate = base / "data" / "raw" / filename
        if candidate.exists():
            return candidate
    raise FileNotFoundError(
        f"Could not find data/raw/{filename} above {here}. "
        "Make sure you're inside your cloned repo (e.g. %cd /content/flyrank-ml-internship)."
    )

CSV_PATH = find_data_csv()
print("Using:", CSV_PATH)
df = pd.read_csv(CSV_PATH)
print("Rows x columns:", df.shape)
print("Distinct clients:", df["client_id"].nunique())
print("trend_direction values (source of the label, never a feature):")
print(df["trend_direction"].value_counts())

## 2. The question: decision, action, cost of a wrong call

**Unit of analysis:** one row = one content page (`content_id`), scored and ranked within its
client (`client_id`). Not a client and not a day — the review decision is made page by page.

**Output:** a ranked queue of pages, each with a priority score, a suggested action
(refresh / expand / protect / monitor), and reason codes a reviewer can inspect before acting
— the same shape as the starter pipeline's `outputs/refresh_queue.csv`.

**Decision this improves:** given a limited review budget, which of a client's existing pages
should be reviewed first this cycle?

**Who acts, and what they do:** a content or SEO reviewer who cannot manually check every page
in a client's portfolio. They work down the ranked queue and, for each flagged page, decide to
refresh, expand, protect, or leave it, guided by the attached reason codes.

**Cost of a wrong call:** the two error types are not symmetric. A false positive — a page
flagged as worth reviewing that turns out fine — costs a reviewer a few minutes, a soft cost.
A false negative — a genuinely declining, high-traffic page that never surfaces — costs the
client silent, ongoing loss of visibility that isn't caught until the next cycle, a harder cost
to recover from. That asymmetry means a ranking that's cautious about missing real decliners is
worth more here than one that only minimizes false alarms.

**Why data/ML helps, and why this isn't just "train a model":** a plain threshold rule (e.g.
"flag anything not updated in 180 days") is cheap but blunt — Section 3 shows it misses most
pages that are actually declining, and a majority of visible pages are declining despite
looking fine on any single field. The signal that separates a page worth reviewing from one
that's fine is spread across several correlated fields (traffic, position, freshness, content
type) in a way that's too tangled to hand-write — that's where a model earns its place over a
rule. But the model is a means to a better queue, not the goal itself: the deliverable a
reviewer uses is the ranked list and its reason codes, not the model file.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Quick look at the data (2-3 real numbers)

Across all **30,000** pages in the starter slice, **54.2%** (16,262 pages) are currently
`trend_direction == "down"` — a clear majority, so "which pages first" is a real, sizeable
problem, not an edge case. Restricting to pages with meaningful search visibility
(impressions_90d ≥ 300, n=18,752) that share climbs to **59.5%** — the pages actually being
seen in search are declining at a higher rate than the portfolio average, which is the
opposite of reassuring. And **14.9%** of all pages (4,467) are both declining *and* stale —
not updated in 90+ days despite still pulling real impressions — which is the kind of page a
simple "time since last update" rule alone would eventually catch, but only after real damage
is already showing in the traffic numbers, and only for the fraction that happens to cross that
one threshold.

These numbers are consistent with the repo's own committed baseline-vs-model comparison in
`outputs/model_report.md`: on this same data, the hand-written rule baseline reaches a
precision@50 of 0.240, while a random forest reaches 0.740 — meaning roughly 12 of the
baseline rule's top 50 flagged pages are real decliners, versus roughly 37 of 50 for the
model. That gap is the concrete evidence that a learned ranking is worth building here.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import pandas as pd
from pathlib import Path

# CSV_PATH / find_data_csv were defined in Section 1's code cell above; re-resolve
# here too so this cell also runs standalone.
try:
    CSV_PATH
except NameError:
    CSV_PATH = find_data_csv()

df = pd.read_csv(CSV_PATH)

# Every row in this slice already has impressions_90d >= 1 and content_age_days >= 90
# (see docs/data-dictionary.md), so this filter mainly documents the assumption rather
# than changing the row count -- kept explicit for clarity and re-use on future slices.
filtered = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)].drop_duplicates(subset="content_id")

n_total = len(filtered)
n_declining = (filtered["trend_direction"] == "down").sum()
pct_declining = round(100 * n_declining / n_total, 1)

visible = filtered[filtered["impressions_90d"] >= 300]
pct_declining_visible = round(100 * (visible["trend_direction"] == "down").sum() / len(visible), 1)

stale_decliners = filtered[
    (filtered["trend_direction"] == "down")
    & (filtered["days_since_last_update"] >= 90)
    & (filtered["impressions_90d"] >= 300)
]
pct_stale_decliners = round(100 * len(stale_decliners) / n_total, 1)

print(f"Total pages: {n_total}")
print(f"Declining pages: {n_declining} ({pct_declining}%)")
print(f"Pages with impressions_90d >= 300: {len(visible)}")
print(f"  ...of which declining: {pct_declining_visible}%")
print(f"Declining AND stale (90+ days since update) AND still visible (>=300 impr): "
      f"{len(stale_decliners)} ({pct_stale_decliners}%)")

## 4. Careful words: what I can and can't claim

**What I can claim:** This is observational, cross-sectional, decision-support work. I can say
a page's freshness, traffic, and position signals are *associated with* whether it's currently
declining, and that a trained ranking *flags/prioritizes* pages at a measured precision@K —
language like "observed," "associated with," and "the model ranks these pages higher" fits the
evidence I have. The output is meant as a reviewer aid: a ranked queue to look at first, with
reason codes attached, not an automatic publishing decision.

**What I can't claim:** I cannot say that refreshing a page *causes* it to recover — that needs
a controlled experiment (e.g. randomizing which flagged pages get refreshed and comparing
outcomes), which this single-snapshot dataset doesn't support. I also can't claim to have
reverse-engineered Google's ranking algorithm, or that the model "understands" content quality
— it's only measuring observable click/impression/session patterns in one portfolio, not the
mechanism behind them. And because `trend_direction`/`trend_pct` define the label itself, I
have to keep them out of the model's features entirely, or any precision number I report would
be measuring leakage, not skill.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.